# pydocs-mcp retrieval methods — a single-repo walkthrough

These notebooks ingest **one real RepoQA repository** ([`marshmallow`](https://github.com/marshmallow-code/marshmallow), copied into `sample_repo/`) and run its **example queries** through each *single-stage* retrieval method, so you can compare what each method retrieves.

Each NL query is a RepoQA *needle*: a description of one function. The goal is to retrieve that **gold** function. We run searches by **building the retrieval pipeline in Python** (the same `RetrieverPipeline` the CLI/MCP server use) — see `nb_helpers.py`.

| # | Notebook | Method | Extra needed |
|---|----------|--------|--------------|
| 1 | `01_bm25` | BM25 keyword (SQLite FTS5) | — |
| 2 | `02_dense_bge` | Dense embeddings (bge-small) | — |
| 3 | `03_dense_qwen3` | Dense embeddings (Qwen3-0.6B) | `[sentence-transformers]` |
| 4 | `04_late_interaction` | Late-interaction / MaxSim (ColBERT) | `[late-interaction]` |
| 5 | `05_tree` | LLM tree reasoning (vectorless) | `OPENAI_API_KEY` |
| 6 | `06_dense_modernbert` | Dense embeddings (gte-modernbert-base, 768-d) | `[sentence-transformers]` |
| 7 | `07_dense_f2llm` | Dense embeddings (F2LLM-v2-0.6B, code) | `[sentence-transformers]` |

## Prerequisites

- **Kernel:** select **`pydocs-mcp (.venv-li)`** (top-right). It has pydocs-mcp + every extra these notebooks use.
- To set up your own env instead:
  ```bash
  pip install -e '.[late-interaction,sentence-transformers]'   # from the repo root
  pip install jupyterlab
  ```
- The **tree** notebook needs an OpenAI key: put `OPENAI_API_KEY=...` in the repo-root `.env` (it is loaded for you).

## How it works

**Indexing (CLI).** Each method notebook indexes `sample_repo/` once with `pydocs-mcp index` into its **own** cache dir (`.pydocs-cache/<method>/`). Why per-method: indexing always builds the keyword (FTS) index *and* dense vectors *and* the document tree, but the **dense vectors depend on the embedder** and late-interaction needs a **separate multi-vector index** — so different methods can't share one index.

**Searching (Python).** We do **not** shell out to `pydocs-mcp search`. Instead `nb_helpers.make_searcher(config, cache_dir)` builds the retrieval pipeline from the method's config and runs each query, returning ranked `Hit`s (rank, score, `qualified_name`).

**Reading results.** `show_results(query, hits)` prints the NL query, the gold function, whether/where the gold appears, and the ranked hits. CLI search renders one merged markdown blob with no scores — running the pipeline ourselves gives the ranked list with scores, which is what we want here.

In [ ]:
# The static example: the committed marshmallow source + its 10 RepoQA needle queries.
from pathlib import Path
from nb_helpers import load_queries

print('sample_repo/ files:')
for p in sorted(Path('sample_repo').rglob('*.py')):
    print('  ', p)

queries = load_queries()
print(f'\n{len(queries)} example queries (RepoQA needles). First two:')
for q in queries[:2]:
    nl = ' '.join(q['query'].split())
    print(f"\n- gold={q['gold_name']}  ({q['needle_path']})")
    print('   ', (nl[:200] + ' …') if len(nl) > 200 else nl)

## Run order

Open `01`–`07` in order; each is self-contained (it indexes, then searches). Re-running an index cell is cheap for this small repo.

> The sample was produced once from the RepoQA dataset and committed so results are reproducible. To regenerate it you'd materialize a repo via `pydocs_eval.datasets.repoqa.RepoQADataset` and copy its `.py` tree + needles — not needed to use these notebooks.